Goal of the code here is to take the big LSTM-formatted datasets that I already generated, read them in, reshape them, and then spit out new files 

In [1]:
# pseudocode

# read file in line by line 
# reformat line to have 3 coordinates
# add to previous event 
# multiply label by line length
# add to previous label 

# write event photons to new csv file (same name, different location) - change the labelling so it matches the exact empir formatting 
# write event labels to new csv file
# make a copy of sources and save to new csv file in new location 

In [3]:
import torch 
import numpy as np
import pandas as pd 
import shutil
import csv

In [3]:
## read files
# label maker and datafile reader for my filename conventions 
def labelmaker(events, density, old, new, basename = None): 
    '''creates labels based on my naming convention for different files, keeps it consistent and easy'''
    base = events + 'ev_n0_es' + density + '.csv'
    if basename: 
        base = basename
    labelbase = 'labels_' + base
    sourcebase = 'sources_' + base

    olddata = old + base
    newdata = new + base 
    sorteddata = new + 'sorted_' + base

    oldlabel = old + labelbase
    newlabel = new + labelbase

    oldsources = old + sourcebase
    newsources = new + sourcebase

    return olddata, newdata, sorteddata, oldlabel, newlabel, oldsources, newsources


In [ ]:
# have all three files open for processing to work through at the same time 

events = '100'
density = '25'
old_preamble = '/home/cgillesp/Downloads/LSTM/'
new_preamble = '/home/cgillesp/Downloads/big_datasets/'

datafile, reform_data, sorteddata, labelfile, reform_labels, sourcefile, sourcecopy = labelmaker(events, density, old_preamble, new_preamble)

data_header = ['x [px]', 'y [px]', 't [s]', 't_relToExtTrigger [s]']
label_header = ['labels']

with open(datafile, newline='') as infile1, \
     open(labelfile, newline='') as infile2, \
     open(reform_data, 'w', newline='') as outfile1, \
     open(reform_labels, 'w', newline='') as outfile2:

    datareader = csv.reader(infile1)
    labelreader = csv.reader(infile2)

    datawriter = csv.writer(outfile1)
    labelwriter = csv.writer(outfile2)

    # Skip original headers (first row of each input file)
    next(datareader)
    next(labelreader)

    # Write custom headers to output files
    datawriter.writerow(data_header)
    labelwriter.writerow(label_header)

    unsorted_events = []

    # Now you're ready to read and process the rows
    for event, label in zip(datareader, labelreader):
        photons = len(event)
        reshaped_event = np.array(event).reshape(-1,3)
        #unnorm_event = np.dot(event, [1e-2, 1e-2, 487/float(events)])
        print(type(reshaped_event[0]))
#         padded_event = ([row.tolist() + ['nan'] for row in event])
#         padded_event = np.array(padded_event)
#         datawriter.writerows(padded_event)
#         labels = np.array([label]*photons)
#         labelwriter.writerows(labels)


# # copying the source file 
# shutil.copyfile(sourcefile, sourcecopy)

# with open(reform_data, newline='') as infile, \
#      open(sorteddata, 'w', newline='') as outfile: 
    
#     reader = csv.reader(infile)
#     rows = list(reader)
#     writer = csv.writer(outfile)
#     writer.writerow(data_header)

#     rows_numeric = []
#     for row in rows:
#         try:
#             rows_numeric.append([float(cell) for cell in row])
#         except ValueError:
#             # If conversion fails (e.g., header row), just keep original row
#             rows_numeric.append(row)

#     # Step 3: Sort rows by the column index, safely handling strings/numbers
#     def sort_key(row):
#         try:
#             return float(row[2])
#         except (ValueError, IndexError):
#             return float('inf')  # Put rows with missing or non-numeric values last

#     sorted_rows = sorted(rows_numeric, key=sort_key)

#     writer.writerows(sorted_rows)


<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy.str_'>
<class 'numpy

In [4]:
def process_file(input_file, output_file, factor, has_header=True):
    '''This one sorts'''
    with open(input_file, 'r', newline='') as infile, open(output_file, 'w', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)

        if has_header:
            header = next(reader)
            writer.writerow(header)

        for row in reader:
            # Multiply the first two columns
            row[0] = str(float(row[0]) * factor)
            row[1] = str(float(row[1]) * factor)

            # Write the modified row
            writer.writerow(row)

In [5]:
def sort(input_file, output_file, has_header=True, numeric=True):
    with open(input_file, 'r', newline='') as infile:
        reader = csv.reader(infile)
        rows = list(reader)

        if has_header:
            header = rows[0]
            data = rows[1:]
        else:
            header = None
            data = rows

        # Sort by the third column (index 2)
        if numeric:
            data.sort(key=lambda x: float(x[2]))
        else:
            data.sort(key=lambda x: x[2])

    # Write the sorted data
    with open(output_file, 'w', newline='') as outfile:
        writer = csv.writer(outfile)
        if header:
            writer.writerow(header)
        writer.writerows(data)

In [98]:
inf = 'big_datasets/plain/sources_10000ev_mixed_n0.csv'
outf = 'big_datasets/empir_formatted/sources_1000ev_mixed_n0.csv'

In [99]:
process_file(inf, outf, 25500)

In [100]:
sort(outf, outf)